# External Validation

**The question:** the model scores about 99% on the paper's dataset. Does it still work
on wheat photos it has never seen, collected by someone else?

**How this notebook answers it.** Your `archive` folder contains the paper's 3,679
images *plus* 631 extra ones from a different source - different file formats
(`.png`, `.jfif`) and different naming (`BrownRust1018.png` instead of
`Brown_rust473.jpg`). Those 631 were never in training. This notebook finds them
automatically by comparing file contents, shows them to the trained model, and reports
how many it gets right.

Nothing is trained here and nothing about the web app changes. It is one pass over 631
images, a few minutes on a normal laptop.

> **Important:** use the model trained on the paper's 3,679 images only (the one your
> Kaggle run produced). If the model was ever trained on the full archive, these 631
> images were part of its training data and the result means nothing.


## Setup

In [ ]:
import os
import hashlib
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix

IMG_SIZE    = (256, 256)
CLASS_NAMES = ['Brown_Rust', 'Healthy', 'Yellow_Rust']
NUM_CLASSES = len(CLASS_NAMES)
BATCH_SIZE  = 32

MODEL_PATH = 'wheat_disease_cnn.keras'      # trained on the paper's dataset

# The paper's dataset - the images the model WAS trained on.
KAGGLE_INPUT = '/kaggle/input/behzad-safari-jalal/data'
if os.path.isdir(KAGGLE_INPUT):
    PAPER_DIR = KAGGLE_INPUT
else:
    import kagglehub
    PAPER_DIR = os.path.join(kagglehub.dataset_download('sinadunk23/behzad-safari-jalal'),
                             'data')

# Your larger collection - the paper's images plus the extra ones. Put the path for
# your machine first if it differs.
ARCHIVE_CANDIDATES = [
    'archive/Dataset',
    r'C:\Users\HP\Downloads\neural-network\archive\Dataset',
]
ARCHIVE_DIR = next((p for p in ARCHIVE_CANDIDATES if os.path.isdir(p)), None)
assert ARCHIVE_DIR, 'Archive folder not found - add its path to ARCHIVE_CANDIDATES.'

print('Paper dataset :', PAPER_DIR)
print('Your archive  :', ARCHIVE_DIR)
print('Model         :', MODEL_PATH, '(found)' if os.path.exists(MODEL_PATH) else '(MISSING)')

## Step 1 - find the images the model has never seen

Every file in both folders is reduced to an md5 fingerprint of its bytes. Anything in
the archive whose fingerprint is absent from the paper's dataset was never used in
training, so it is fair game as a test image.

In [ ]:
IMG_EXTS = ('.jpg', '.jpeg', '.jfif', '.png', '.bmp', '.gif')


def all_images(root):
    """Every image file anywhere under `root`."""
    for parent, _, files in os.walk(root):
        for name in sorted(files):
            if name.lower().endswith(IMG_EXTS):
                yield os.path.join(parent, name)


def file_md5(path):
    with open(path, 'rb') as fh:
        return hashlib.md5(fh.read()).hexdigest()


def canonical_class(folder_name):
    """'Brown rust' -> 'Brown_Rust'. Returns None for anything unrecognised."""
    wanted = folder_name.lower().replace('_', ' ')
    for name in CLASS_NAMES:
        if name.lower().replace('_', ' ') == wanted:
            return name
    return None


trained_on = {file_md5(p) for p in all_images(PAPER_DIR)}
print(f'{len(trained_on)} images in the paper dataset (the model saw these)\n')

unseen_paths, unseen_labels = [], []
for path in all_images(ARCHIVE_DIR):
    if file_md5(path) in trained_on:
        continue
    label = canonical_class(os.path.basename(os.path.dirname(path)))
    if label is None:
        continue
    unseen_paths.append(path)
    unseen_labels.append(label)

counts = defaultdict(int)
for label in unseen_labels:
    counts[label] += 1

print(f"{'Class':14s}{'unseen images':>15s}")
print('-' * 29)
for name in CLASS_NAMES:
    print(f'{name:14s}{counts[name]:15d}')
print('-' * 29)
print(f"{'TOTAL':14s}{len(unseen_paths):15d}")

## Step 2 - show them to the model

Each image goes through exactly the same preparation as during training: convert to RGB,
resize to 256x256, scale the pixels to 0-1.

In [ ]:
model = tf.keras.models.load_model(MODEL_PATH)


def load_batch(paths):
    """Load and prepare a list of images the same way training did."""
    images = [np.asarray(Image.open(p).convert('RGB').resize(IMG_SIZE), dtype='float32') / 255.0
              for p in paths]
    return np.stack(images)


predictions, confidences = [], []
for start in range(0, len(unseen_paths), BATCH_SIZE):
    batch = load_batch(unseen_paths[start:start + BATCH_SIZE])
    probs = model.predict(batch, verbose=0)
    predictions.extend(CLASS_NAMES[i] for i in probs.argmax(axis=1))
    confidences.extend(probs.max(axis=1))
    print(f'\r{start + len(batch)} / {len(unseen_paths)} images', end='')

print('\n\nDone.')

## Step 3 - the result

In [ ]:
correct = sum(p == t for p, t in zip(predictions, unseen_labels))
accuracy = correct / len(unseen_labels)

print(f'Unseen images     : {len(unseen_labels)}')
print(f'Correct           : {correct}')
print(f'External accuracy : {accuracy:.4f}   ({accuracy:.1%})')
print(f'Mean confidence   : {np.mean(confidences):.1%}')
print()
print(classification_report(unseen_labels, predictions,
                            labels=CLASS_NAMES, digits=3, zero_division=0))

cm = confusion_matrix(unseen_labels, predictions, labels=CLASS_NAMES)
fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right')
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(CLASS_NAMES)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('External test set (unseen images)')
fig.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()

## Step 4 - compare with the reported accuracy

In [ ]:
PAPER_VAL_ACCURACY = 0.9900     # what the main notebook reported on its own split

print(f'On the paper dataset (validation split): {PAPER_VAL_ACCURACY:.1%}')
print(f'On {len(unseen_labels)} unseen images from another source: {accuracy:.1%}')
print(f'Difference: {(accuracy - PAPER_VAL_ACCURACY) * 100:+.1f} percentage points\n')

drop = PAPER_VAL_ACCURACY - accuracy
if drop < 0.02:
    print('The accuracy holds up on images from a different source.')
    print('That is a stronger claim than the paper makes, and worth stating.')
elif drop < 0.15:
    print('A moderate drop. The model generalises, but not as well as the headline')
    print('number suggests. Report both numbers, not just the higher one.')
else:
    print('A large drop. The reported accuracy is specific to this dataset and does')
    print('not transfer to photos collected elsewhere - the main finding of this test.')

## What to write

Report **both** numbers together, never the higher one alone:

> *"The replicated CNN reached 99.0% on the paper's validation split. Evaluated on 631
> images from a different source that were absent from training, accuracy was X%."*

Then say what the confusion matrix shows - which class survives the change of source and
which one collapses. Healthy images from a different photographer are usually the first
to break, because "healthy leaf" looks different under a different camera and background.

**State the limitations honestly.** The labels of these 631 images come from the folder
names you inherited, not from a plant pathologist, and some of the Healthy ones have
random filenames suggesting they were collected from the web. An examiner will ask; say
it before they do.
